# Column Transformer in Machine Learning | How to use ColumnTransformer in Sklearn
ColumnTransformer is a scikit-learn tool that lets you apply different preprocessing steps to different columns simultaneously, in a single, organized step — instead of manually transforming each column type one at a time and then trying to stitch them back together.
The problem it solves

**Think about a real dataset like Titanic:**

- Age (numerical) → needs Standardization
- Fare (numerical) → needs Standardization  
- Gender (nominal) → needs One-Hot Encoding
- Education (ordinal) → needs Ordinal Encoding

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [3]:
df = pd.read_csv('covid_toy.csv')

In [5]:
df.head(5)

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [7]:
df.shape

(100, 6)

In [9]:
df.describe()

,age,fever
count,100.000000,90.000000
mean,44.220000,100.844444
std,24.878931,2.054926
min,5.000000,98.000000
25%,20.000000,99.000000
50%,45.000000,101.000000
75%,66.500000,102.750000
max,84.000000,104.000000


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        100 non-null    int64  
 1   gender     100 non-null    object 
 2   fever      90 non-null     float64
 3   cough      100 non-null    object 
 4   city       100 non-null    object 
 5   has_covid  100 non-null    object 
dtypes: float64(1), int64(1), object(4)
memory usage: 4.8+ KB


In [18]:
df['cough'].value_counts()


cough
Mild      62
Strong    38
Name: count, dtype: int64

In [19]:
df['city'].value_counts()

city
Kolkata      32
Bangalore    30
Delhi        22
Mumbai       16
Name: count, dtype: int64

In [20]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [21]:
from sklearn.model_selection import train_test_split
xtrain, xtest, ytrain, ytest = train_test_split(df.drop(columns=['has_covid']), df['has_covid'],test_size=0.2)

In [22]:
xtrain

,age,gender,fever,cough,city
96,51,Female,101.0,Strong,Kolkata
45,72,Male,99.0,Mild,Bangalore
83,17,Female,104.0,Mild,Kolkata
99,10,Female,98.0,Strong,Kolkata
7,20,Female,NaN,Strong,Mumbai
...,...,...,...,...,...
28,16,Male,104.0,Mild,Kolkata
42,27,Male,100.0,Mild,Delhi
81,65,Male,99.0,Mild,Delhi
14,51,Male,104.0,Mild,Bangalore


### Without Transformer

In [30]:
# Fill out missing values
si = SimpleImputer()
xtrain_fever = si.fit_transform(xtrain[['fever']])
xtest_fever = si.fit_transform(xtest[['fever']])
xtrain_fever.shape

(80, 1)

In [32]:
# Ordinalencoding -> cough
oe = OrdinalEncoder(categories=[['Mild','Strong']])
xtrain_cough = oe.fit_transform(xtrain[['cough']])

# also the test data
xtest_cough = oe.fit_transform(xtest[['cough']])

xtrain_cough.shape

(80, 1)

In [36]:
# OneHotEncoding -> gender,city
ohe = OneHotEncoder(drop='first')
xtrain_gender_city = ohe.fit_transform(xtrain[['gender','city']])

# also the test data
xtest_gender_city = ohe.fit_transform(xtest[['gender','city']])

xtrain_gender_city.shape

(80, 4)

In [39]:
# Extracting Age
xtrain_age = xtrain.drop(columns=['gender','fever','cough','city']).values

# also the test data
xtest_age = xtest.drop(columns=['gender','fever','cough','city']).values

xtrain_age.shape

(80, 1)

In [40]:
xtrain_transformed = np.concatenate((xtrain_age,xtrain_fever,xtrain_gender_city,xtrain_cough),axis=1)
# also the test data
xtest_transformed = np.concatenate((xtest_age,xtest_fever,xtest_gender_city,xtest_cough),axis=1)

xtrain_transformed.shape

ValueError: all the input arrays must have same number of dimensions, but the array at index 0 has 2 dimension(s) and the array at index 2 has 0 dimension(s)

In [41]:
from sklearn.compose import ColumnTransformer

In [43]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first'),['gender','city'])
],remainder='passthrough')

In [45]:
transformer.fit_transform(xtrain).shape

(80, 7)

In [46]:
transformer.transform(xtest).shape

(20, 7)